# Data Construction for binary classification (XP Coefficients)

We build `out_data/binary_vosa_XPcoeff_110d_L2.csv` (and a matching `.npz`) from VOSA labels and Gaia XP coefficient data.


1. Load `data/VOSA_labels_training.csv` and keep `source_id` + `label_raw` (which are already 0/1).
2. Collect Gaia XP coefficient VOTables from `out_data/` (files named `sjs_*_result.vot`).
   If those .vot files are missing, we can call the Gaia AIP to download them.
3. For each adequate source_id, read `bp_coefficients` and `rp_coefficients` arrays.
   Each is padded/truncated to length 55, then concatenated into a 110‑D feature vector.
4. Merge features with VOSA labels on `source_id`.
5. L2‑normalize the 110‑D features and save:
   - `out_data/binary_vosa_XPcoeff_110d_L2.csv` for ML methods
   - `out_data/binary_vosa_XPcoeff_110d.npz` for faster loading, includes raw + L2

**Expected file structure** (from root):
- `data/VOSA_labels_training.csv`
- `out_data/sjs_*_result.vot` (cached XP coefficient VOTables)

Label CSV must be downloaded from:

```
https://github.com/mambr-astro/Hot_sds_GaiaXP/tree/main/data
```


In [1]:
import os
import io
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import pyvo as vo
from astropy.table import Table


In [ ]:
# Paths
ROOT = Path.cwd()
if not (ROOT / 'data' / 'VOSA_labels_training.csv').exists():
    hits = list(ROOT.rglob('data/VOSA_labels_training.csv'))
    if hits:
        ROOT = hits[0].parents[1]
    else:
        raise FileNotFoundError('Could not find data/VOSA_labels_training.csv; download it from the aforementioned github repo')

DATA_DIR = ROOT / 'data'
OUT_DIR = ROOT / 'out_data'
OUT_DIR.mkdir(exist_ok=True)

print('ROOT:', ROOT)
print('DATA_DIR:', DATA_DIR)
print('OUT_DIR:', OUT_DIR)


In [ ]:
# Load VOSA labels (binary classification)
vosa_path = DATA_DIR / 'VOSA_labels_training.csv'
df_vosa = pd.read_csv(vosa_path)

# Normalize column names
if 'GaiaDR3' in df_vosa.columns and 'source_id' not in df_vosa.columns:
    df_vosa = df_vosa.rename(columns={'GaiaDR3': 'source_id'})
if 'VOSA' in df_vosa.columns and 'label_raw' not in df_vosa.columns:
    df_vosa = df_vosa.rename(columns={'VOSA': 'label_raw'})

df_vosa = df_vosa[['source_id', 'label_raw']].dropna()
df_vosa['source_id'] = df_vosa['source_id'].astype('int64')
df_vosa['y'] = df_vosa['label_raw'].astype(int)

source_ids = df_vosa['source_id'].unique().tolist()
print('labels rows:', len(df_vosa), '| unique source_ids:', len(source_ids))
print('class balance:', df_vosa['y'].value_counts().to_dict())


In [ ]:
# Optional: download XP coefficient VOTables via Gaia AIP if missing
vot_files = sorted(OUT_DIR.glob('sjs_*_result.vot'))
if not vot_files:
    GAIA_AIP_TOKEN = os.getenv('GAIA_AIP_TOKEN', 'YOUR_API_TOKEN_GOES_HERE').strip()
    #                                             ^^^^^^^^^^^^^^^^^^^^^^^^^
    if (not GAIA_AIP_TOKEN) or any(ord(c) > 127 for c in GAIA_AIP_TOKEN):
        raise RuntimeError('Missing GAIA_AIP_TOKEN (ASCII only). Set it or download VOTables separately.')

    TAP_URL = 'https://gaia.aip.de/tap'
    SJS_URL = 'https://gaia.aip.de/uws/simple-join-service'

    sess = requests.Session()
    sess.headers['Authorization'] = GAIA_AIP_TOKEN if GAIA_AIP_TOKEN.startswith('Token ') else f'Token {GAIA_AIP_TOKEN}'

    def tap_create_async_job(query: str, runid: str) -> str:
        url = f'{TAP_URL}/async'
        payload = {
            'REQUEST': 'doQuery',
            'LANG': 'ADQL',
            'FORMAT': 'votable',
            'QUERY': query.strip().rstrip(';'),
            'RUNID': runid,
        }
        r = sess.post(url, data=payload, allow_redirects=False, timeout=120)
        if r.status_code in (302, 303) and 'Location' in r.headers:
            job_url = r.headers['Location']
            if job_url.startswith('/'):
                job_url = 'https://gaia.aip.de' + job_url
        else:
            raise RuntimeError(f'Unexpected TAP async response: HTTP {r.status_code}; body={r.text[:300]!r}')
        job_id = job_url.rstrip('/').split('/')[-1]
        sess.post(f'{job_url}/phase', data={'PHASE': 'RUN'}, timeout=60).raise_for_status()
        t0 = time.time()
        while True:
            ph = sess.get(f'{job_url}/phase', timeout=60).text.strip()
            if ph in ('COMPLETED', 'ERROR', 'ABORTED'):
                break
            if time.time() - t0 > 180:
                raise TimeoutError('TAP async job timed out (>180s).')
            time.sleep(1.5)
        if ph != 'COMPLETED':
            raise RuntimeError(f'TAP async job ended with phase={ph}')
        return job_id

    def sjs_download(job_id: str, join_table: str) -> Path:
        service = vo.dal.DALService(SJS_URL, session=sess)
        q = service.create_query(
            job_id=job_id,
            column_name='source_id',
            responseformat='votable',
            join_table=join_table,
            data_structure='COMBINED',
        )
        resp = q.submit(post=True)
        resp.raise_for_status()
        job = vo.io.uws.parse_job(io.BytesIO(resp.content))
        service._session.post(f'{service._baseurl}/{job.jobid}/phase', data={'PHASE': 'RUN'}, stream=True).raise_for_status()
        t0 = time.time()
        while True:
            ph = service._session.get(f'{service._baseurl}/{job.jobid}/phase').text.strip()
            if ph in ('COMPLETED', 'ERROR', 'ABORTED'):
                break
            if time.time() - t0 > 240:
                raise TimeoutError('SJS job timed out (>240s).')
            time.sleep(1.5)
        if ph != 'COMPLETED':
            raise RuntimeError(f'SJS ended with phase={ph}')
        job_url = f'{service._baseurl}/{job.jobid}'
        job2 = vo.io.uws.parse_job(io.BytesIO(service._session.get(job_url).content))
        results = job2.results
        if hasattr(results, 'keys') and callable(getattr(results, 'keys')):
            first_key = sorted(list(results.keys()))[0]
            href = results[first_key].href
            key = str(first_key)
        else:
            res_list = list(results)
            if not res_list:
                raise RuntimeError('SJS job has no results.')
            href = res_list[0].href
            key = 'result'
        out_path = OUT_DIR / f'sjs_{job2.jobid}_{key}.vot'
        out_path.write_bytes(service._session.get(href, timeout=300).content)
        return out_path

    CHUNK = 400
    join_table = 'gaiadr3.xp_continuous_mean_spectrum'
    for i in range(0, len(source_ids), CHUNK):
        chunk_ids = source_ids[i:i+CHUNK]
        ids_sql = ','.join(str(int(sid)) for sid in chunk_ids)
        query = f'SELECT source_id FROM gaiadr3.gaia_source WHERE source_id IN ({ids_sql})'
        job_id = tap_create_async_job(query, runid=f'ids_for_sjs_{i//CHUNK:04d}')
        vot_path = sjs_download(job_id, join_table)
        print('Saved:', vot_path)

    vot_files = sorted(OUT_DIR.glob('sjs_*_result.vot'))

print('XP VOTables found:', len(vot_files))


In [ ]:
# Load cached Gaia XP continuous coefficient VOTables
vot_files = sorted(OUT_DIR.glob('sjs_*_result.vot'))
if not vot_files:
    raise RuntimeError('No cached XP VOTables in out_data/. Download via the Gaia API cell above.')

def _ensure_source_id(df: pd.DataFrame) -> pd.DataFrame:
    if 'source_id' in df.columns:
        df['source_id'] = df['source_id'].astype('int64')
        return df
    if 'datalinkID' in df.columns:
        sid = df['datalinkID'].astype(str).str.extract(r'(\d{10,})')[0]
        if sid.isna().any():
            raise KeyError('Cannot parse source_id from datalinkID.')
        df = df.copy()
        df['source_id'] = sid.astype('int64')
        return df
    raise KeyError(f'No source_id column. Columns: {list(df.columns)}')

def _as_float_array(x):
    if isinstance(x, np.ndarray):
        return x.astype(float, copy=False).ravel()
    if isinstance(x, (list, tuple)):
        return np.asarray(x, dtype=float).ravel()
    s = str(x).strip()
    if s.startswith('[') and s.endswith(']'):
        items = [t for t in s[1:-1].split(',') if t.strip() != '']
        return np.asarray([float(t) for t in items], dtype=float).ravel()
    raise TypeError(f'Cannot parse array from {type(x)}')

def _pad_or_truncate(a, L=55):
    a = np.asarray(a, dtype=float).ravel()
    if len(a) == L:
        return a
    if len(a) > L:
        return a[:L]
    out = np.zeros(L, dtype=float)
    out[:len(a)] = a
    return out

dfs = []
for p in vot_files:
    df = Table.read(p).to_pandas()
    df = _ensure_source_id(df)
    dfs.append(df)

df_xp_raw = pd.concat(dfs, ignore_index=True)
df_xp_raw = df_xp_raw.drop_duplicates('source_id')
df_xp_raw = df_xp_raw[df_xp_raw['source_id'].isin(source_ids)]

# Ensure every labeled source_id has a matching XP row
expected_ids = set(source_ids)
got_ids = set(df_xp_raw["source_id"].tolist())
missing = sorted(list(expected_ids - got_ids))
extra = sorted(list(got_ids - expected_ids))
if missing or extra or (not df_xp_raw["source_id"].is_unique):
    raise RuntimeError("Source_id mismatch between labels and XP data.")

bp_col = 'bp_coefficients'
rp_col = 'rp_coefficients'
if bp_col not in df_xp_raw.columns or rp_col not in df_xp_raw.columns:
    raise KeyError(f'Missing coefficient columns. Have: {list(df_xp_raw.columns)}')

bp = df_xp_raw[bp_col].map(_as_float_array).map(lambda a: _pad_or_truncate(a, 55))
rp = df_xp_raw[rp_col].map(_as_float_array).map(lambda a: _pad_or_truncate(a, 55))

BP = np.vstack(bp.values)
RP = np.vstack(rp.values)
X = np.hstack([BP, RP]).astype(np.float32)


print('XP rows:', len(df_xp_raw), '| X shape:', X.shape)


In [ ]:
# Build final feature table and save
df_feat = pd.DataFrame(X, columns=[f'c{i:03d}' for i in range(X.shape[1])])
df_feat.insert(0, 'source_id', df_xp_raw['source_id'].values)
df_feat = df_feat.merge(df_vosa[['source_id', 'y']], on='source_id', how='inner')

# Sanity check: all labels matched
if len(df_feat) != len(source_ids):
    raise RuntimeError(f'Mismatch after merge: got {len(df_feat)} rows, expected {len(source_ids)}.')

y = df_feat['y'].astype(int).to_numpy()
source_id = df_feat['source_id'].astype('int64').to_numpy()
X = df_feat.drop(columns=['source_id', 'y']).to_numpy(dtype=np.float32)

norms = np.linalg.norm(X, axis=1, keepdims=True)
if (norms.squeeze() == 0).any():
    raise RuntimeError('Found zero-norm rows in X.')
X_l2 = X / norms

np.savez_compressed(
    OUT_DIR / 'binary_vosa_XPcoeff_110d.npz',
    X=X,
    X_l2=X_l2,
    y=y,
    source_id=source_id,
)

df_out = pd.DataFrame(X_l2, columns=[f'c{i:03d}' for i in range(X_l2.shape[1])])
df_out.insert(0, 'source_id', source_id)
df_out.insert(1, 'y', y)
df_out.to_csv(OUT_DIR / 'binary_vosa_XPcoeff_110d_L2.csv', index=False)

print('Saved:')
print(' -', OUT_DIR / 'binary_vosa_XPcoeff_110d.npz')
print(' -', OUT_DIR / 'binary_vosa_XPcoeff_110d_L2.csv')
